In [4]:
pip install seaborn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import json
import webbrowser
from pathlib import Path

import numpy as np
import pandas as pd


DATA_PATH = "features_dataset.csv"
OUTPUT_PATH = "correlation_matrix.html"

HTML = """<!DOCTYPE html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<title>Correlation Matrix</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body {
    font-family: 'IBM Plex Mono', 'Fira Code', monospace;
    background: #0d1117;
    color: #c9d1d9;
    min-height: 100vh;
}
.header {
    background: #161b22;
    border-bottom: 1px solid #21262d;
    padding: 20px 28px;
    position: sticky;
    top: 0;
    z-index: 10;
}
h1 { font-size: 18px; color: #e6edf3; font-weight: 700; margin-bottom: 14px; }
.controls { display: flex; flex-wrap: wrap; gap: 16px; align-items: flex-end; margin-bottom: 12px; }
.group { display: flex; flex-direction: column; gap: 6px; }
.label { font-size: 10px; color: #8b949e; letter-spacing: 0.8px; text-transform: uppercase; }
.btn-row { display: flex; gap: 4px; }
button {
    padding: 4px 10px;
    border-radius: 6px;
    border: 1px solid #30363d;
    background: transparent;
    color: #8b949e;
    cursor: pointer;
    font-size: 12px;
    font-family: inherit;
    transition: all 0.1s;
}
button.active { border-color: #388bfd; background: #1f6feb33; color: #79c0ff; }
input {
    background: #0d1117;
    border: 1px solid #30363d;
    border-radius: 6px;
    color: #c9d1d9;
    padding: 5px 12px;
    font-size: 12px;
    font-family: inherit;
    outline: none;
    width: 220px;
}
.stats { display: flex; gap: 24px; margin-top: 10px; align-items: baseline; }
.stat-val { font-size: 18px; font-weight: 700; margin-right: 4px; }
table { width: 100%; border-collapse: collapse; font-size: 12px; }
thead tr { background: #161b22; border-bottom: 2px solid #21262d; }
th {
    padding: 10px 14px;
    text-align: left;
    font-size: 10px;
    font-weight: 600;
    color: #8b949e;
    letter-spacing: 0.8px;
    text-transform: uppercase;
    cursor: pointer;
    user-select: none;
}
th:hover { color: #c9d1d9; }
td { padding: 8px 14px; vertical-align: middle; }
tr.even { background: #0d1117; }
tr.odd { background: #0f1318; }
tr:hover td { background: #1c2128 !important; }
.feat { color: #79c0ff; font-weight: 500; }
.r-cell { text-align: center; font-weight: 700; font-size: 13px; width: 80px; }
.bar-wrap { display: flex; align-items: center; gap: 6px; width: 200px; }
.bar-track {
    flex: 1; height: 6px; background: #21262d;
    border-radius: 3px; position: relative; overflow: hidden;
}
.bar-center {
    position: absolute; top: 0; left: 50%;
    width: 1px; height: 100%; background: #30363d;
}
.bar-fill { position: absolute; top: 0; height: 100%; border-radius: 3px; }
.bar-pct { font-size: 10px; color: #8b949e; width: 32px; text-align: right; }
.idx { color: #484f58; width: 48px; }
.empty { padding: 40px; text-align: center; color: #484f58; }
</style>
</head>
<body>
<div class="header">
  <h1>Correlation Matrix &mdash; TOTAL_PAIRS pairs / TOTAL_FEATURES features</h1>
  <div class="controls">
    <div class="group">
      <span class="label">Threshold |r|</span>
      <div class="btn-row" id="thr-group">
        <button onclick="setThr(0.3)" data-t="0.3">0.3</button>
        <button onclick="setThr(0.4)" data-t="0.4">0.4</button>
        <button onclick="setThr(0.5)" data-t="0.5" class="active">0.5</button>
        <button onclick="setThr(0.6)" data-t="0.6">0.6</button>
        <button onclick="setThr(0.7)" data-t="0.7">0.7</button>
        <button onclick="setThr(0.8)" data-t="0.8">0.8</button>
        <button onclick="setThr(0.9)" data-t="0.9">0.9</button>
      </div>
    </div>
    <div class="group">
      <span class="label">Filter</span>
      <input id="search" placeholder="Feature name..." oninput="render()">
    </div>
    <div class="group">
      <span class="label">Sort</span>
      <div class="btn-row" id="sort-group">
        <button onclick="setSort('abs')" data-s="abs" class="active">By |r|</button>
        <button onclick="setSort('pos')" data-s="pos">Positive</button>
        <button onclick="setSort('neg')" data-s="neg">Negative</button>
      </div>
    </div>
  </div>
  <div class="stats">
    <div><span class="stat-val" id="cnt" style="color:#e6edf3">0</span><span class="label">Shown</span></div>
    <div><span class="stat-val" id="pos" style="color:#3fb950">0</span><span class="label">Positive</span></div>
    <div><span class="stat-val" id="neg" style="color:#f85149">0</span><span class="label">Negative</span></div>
  </div>
</div>

<table>
  <thead>
    <tr>
      <th class="idx">#</th>
      <th onclick="setSort('a')">Feature A</th>
      <th onclick="setSort('b')">Feature B</th>
      <th onclick="setSort('abs')" style="text-align:center">r</th>
      <th>Strength</th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>

<script>
const DATA = PAIRS_DATA;
let thr = 0.5, search = "", sortKey = "abs";

function setThr(t) {
    thr = t;
    document.querySelectorAll("#thr-group button").forEach(
        b => b.classList.toggle("active", parseFloat(b.dataset.t) === t)
    );
    render();
}

function setSort(s) {
    sortKey = s;
    document.querySelectorAll("#sort-group button").forEach(
        b => b.classList.toggle("active", b.dataset.s === s)
    );
    render();
}

document.getElementById("search").addEventListener("input", e => {
    search = e.target.value.toLowerCase();
    render();
});

function color(r) {
    const a = Math.abs(r);
    if (r > 0) {
        return `rgb(${Math.round(220*a+100*(1-a))},${Math.round(10*a+60*(1-a))},${Math.round(10*a+60*(1-a))})`;
    }
    return `rgb(${Math.round(20*a+30*(1-a))},${Math.round(30*a+80*(1-a))},${Math.round(220*a+80*(1-a))})`;
}

function bg(r) {
    const a = Math.abs(r).toFixed(2);
    return r > 0 ? `rgba(220,60,60,${(Math.abs(r)*0.22).toFixed(2)})` : `rgba(60,100,220,${(Math.abs(r)*0.22).toFixed(2)})`;
}

function bar(r) {
    const pct = Math.abs(r) * 50;
    const left = r > 0 ? 50 : 50 - pct;
    const c = color(r);
    return `<div class="bar-wrap">
      <div class="bar-track">
        <div class="bar-center"></div>
        <div class="bar-fill" style="left:${left}%;width:${pct}%;background:${c};opacity:0.75"></div>
      </div>
      <span class="bar-pct">${Math.round(Math.abs(r)*100)}%</span>
    </div>`;
}

function render() {
    let rows = DATA.filter(d => Math.abs(d.r) >= thr);
    if (search) rows = rows.filter(d => d.a.includes(search) || d.b.includes(search));

    const sorters = {
        abs: (x, y) => Math.abs(y.r) - Math.abs(x.r),
        pos: (x, y) => y.r - x.r,
        neg: (x, y) => x.r - y.r,
        a:   (x, y) => x.a.localeCompare(y.a),
        b:   (x, y) => x.b.localeCompare(y.b),
    };
    rows.sort(sorters[sortKey] || sorters.abs);

    document.getElementById("cnt").textContent = rows.length;
    document.getElementById("pos").textContent = rows.filter(d => d.r > 0).length;
    document.getElementById("neg").textContent = rows.filter(d => d.r < 0).length;

    document.getElementById("tbody").innerHTML = rows.length === 0
        ? `<tr><td colspan="5" class="empty">No pairs with |r| >= ${thr}</td></tr>`
        : rows.map((d, i) => `<tr class="${i % 2 === 0 ? "even" : "odd"}">
            <td class="idx">${i + 1}</td>
            <td class="feat">${d.a}</td>
            <td class="feat">${d.b}</td>
            <td class="r-cell" style="color:${color(d.r)};background:${bg(d.r)}">${d.r > 0 ? "+" : ""}${d.r.toFixed(3)}</td>
            <td>${bar(d.r)}</td>
          </tr>`).join("");
}

render();
</script>
</body>
</html>"""


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["datetime"], index_col="datetime")
    return df.select_dtypes(include=[np.number])


def compute_pairs(df: pd.DataFrame) -> list:
    corr = df.corr().round(3)
    cols = corr.columns.tolist()
    pairs = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            r = float(corr.iloc[i, j])
            if not np.isnan(r):
                pairs.append({"a": cols[i], "b": cols[j], "r": round(r, 3)})
    pairs.sort(key=lambda x: abs(x["r"]), reverse=True)
    return pairs


def build_html(pairs: list, n_features: int) -> str:
    html = HTML.replace("TOTAL_PAIRS", str(len(pairs)))
    html = html.replace("TOTAL_FEATURES", str(n_features))
    html = html.replace("PAIRS_DATA", json.dumps(pairs, separators=(",", ":")))
    return html


if __name__ == "__main__":
    df = load_data(DATA_PATH)
    pairs = compute_pairs(df)
    html = build_html(pairs, n_features=len(df.columns))
    Path(OUTPUT_PATH).write_text(html, encoding="utf-8")
    print(f"Saved: {OUTPUT_PATH}  ({len(pairs)} pairs, {len(df.columns)} features)")
    webbrowser.open(OUTPUT_PATH)

Saved: correlation_matrix.html  (7260 pairs, 122 features)
